# Dragapult policy prior v2 — winners + top-team/lean4

Bu notebook **bizim ham replay datasetimizi** kullanır ama önceki prior’dan farklı olarak:
- sadece **Dragapult oynayan ve kazanan** tarafların aksiyonlarını alır,
- mümkünse sadece `lean4` ve `top-team` klasörlerini kullanır,
- Mazlum’un option feature/card embedding/masked CE hattını kullanır.

Amaç: sample/Lucario’yu bozan eski/kötü Dragapult aksiyonlarını ayıklamak.

## Gerekli datasetler
- `ptcg-replays` (ham JSON/zip)
- `ptcg-code` (bu zip: `ptcg-code.zip`)

Çıktı: `/kaggle/working/policy_prior_v2.npz`


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
from collections import Counter
import numpy as np

INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working'); WORK.mkdir(parents=True,exist_ok=True)
# extract code zip if needed
code_zips=sorted(INPUT.rglob('ptcg-code.zip'))
code_root=WORK/'ptcg-code'
if code_zips and not (code_root/'src/mazlum/build_dataset.py').exists():
    print('extracting code',code_zips[0])
    with zipfile.ZipFile(code_zips[0]) as zf: zf.extractall(code_root)
builders=[]
for root in (code_root, INPUT):
    if root.exists(): builders += list(root.rglob('src/mazlum/build_dataset.py'))
if not builders: raise RuntimeError('build_dataset.py yok; ptcg-code.zip ekle.')
BUILD=builders[0]; CODE_ROOT=BUILD.parents[2]
TRAIN=None
for root in (code_root, INPUT):
    if root.exists():
        TRAIN=next(iter(root.rglob('src/mazlum/train_bc.py')),None) or next(iter(root.rglob('train_bc.py')),None)
        if TRAIN: break
if not TRAIN: raise RuntimeError('train_bc.py yok.')
print('BUILD',BUILD); print('TRAIN',TRAIN)

# raw replay extraction
def find_eps(root): return list(root.rglob('episode-*.json'))
eps=find_eps(INPUT)
if eps:
    REPLAYS=INPUT
else:
    REPLAYS=WORK/'raw_replays'; REPLAYS.mkdir(parents=True,exist_ok=True)
    for z in sorted(INPUT.rglob('*.zip')):
        print('extracting replay',z)
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if not m.lower().endswith('.json') or m.endswith('/'): continue
                d=REPLAYS/Path(m)
                d.parent.mkdir(parents=True,exist_ok=True)
                if d.exists(): continue
                with zf.open(m) as s, open(d,'wb') as t: shutil.copyfileobj(s,t,1024*1024)
print('raw episodes',len(find_eps(REPLAYS)))


In [ ]:
# Filter: Dragapult players, winners, preferably lean4/top-team folders.
WINNERS_ONLY=True
SOURCE_FILTER=True
SOURCES={'lean_57629056','top_oct24','replays'}  # fallback below if none
OUR='Attention Is All You Mew'

def extract_decks(e):
    try:
        a=e['steps'][0][0]['visualize'][0]['action']
        return [list(map(int,d)) for d in a] if isinstance(a,list) and len(a)==2 else None
    except (KeyError,IndexError,TypeError): return None
def is_dp(d): return Counter(d).get(121,0)>=2

selected=WORK/'prior_v2_replays'; selected.mkdir(parents=True,exist_ok=True)
allowed={}
all_paths=list(REPLAYS.rglob('episode-*.json'))
paths=[q for q in all_paths if q.parent.name in SOURCES] if SOURCE_FILTER else all_paths
if not paths:
    print('source filter empty; using all replay paths')
    paths=all_paths
for pth in paths:
    try: e=json.loads(pth.read_text(encoding='utf-8'))
    except Exception: continue
    decks=extract_decks(e); rewards=e.get('rewards') or []
    if not decks or len(rewards)!=2: continue
    players=[i for i,d in enumerate(decks) if is_dp(d)]
    if WINNERS_ONLY: players=[i for i in players if rewards[i]==1]
    if not players: continue
    eid=(e.get('info') or {}).get('EpisodeId')
    if eid is None: continue
    eid=int(eid)
    dest=selected/f'episode-{eid}-replay.json'
    if not dest.exists():
        try: os.symlink(pth,dest)
        except OSError: shutil.copy2(pth,dest)
    allowed[eid]=players
print('selected episodes',len(allowed),'files',len(list(selected.glob('*.json'))))
if not allowed:
    raise RuntimeError('Filtre sonucu boş; SOURCES/WINNERS_ONLY değerlerini gevşet.')

## Dataset build + sadece seçilen Dragapult oyuncu satırları

In [ ]:
raw=WORK/'dataset_prior_v2_all.npz'
cmd=[sys.executable,str(BUILD),'--replays',str(selected),'--out',str(raw),'--val-fraction','0.2']
print(' '.join(cmd)); subprocess.run(cmd,check=True)

d=np.load(raw,mmap_mode='r'); rows={k:d[k] for k in d.files}
n=rows['player'].shape[0]
keep=np.array([int(rows['player'][i]) in allowed.get(int(rows['episode_id'][i]),[]) for i in range(n)],dtype=bool)
out=WORK/'dataset_prior_v2_ours.npz'
np.savez_compressed(out,**{k:v[keep] for k,v in rows.items()})
print('rows',n,'keep',int(keep.sum()),'MB',round(out.stat().st_size/1e6,2))

## Train policy prior v2

In [ ]:
prior=WORK/'policy_prior_v2.npz'
cmd=[sys.executable,str(TRAIN),'--dataset',str(out),'--out',str(prior),'--epochs','10','--batch-size','64','--lr','0.001','--arena-every','0']
print(' '.join(cmd)); subprocess.run(cmd,check=True)

In [ ]:
from IPython.display import FileLink,display
print('exists',prior.exists(),'MB',round(prior.stat().st_size/1e6,2) if prior.exists() else None)
if prior.exists():
    with np.load(prior) as z: print('arrays',len(z.files),'params',sum(z[k].size for k in z.files))
    display(FileLink(str(prior)))

## Not
Bu prior doğrudan submission değil. Sonraki adımda lean4 search’e **sadece Alakazam/Garchomp gibi prior’ın iyi olduğu matchup’larda** bağlanacak; sample/Lucario’da kapalı kalacak.
